In [705]:
!pip install category_encoders

What parameter value gives more weight to Recall than to Precision?

β = 1 → F1 Score (равный вес)

β > 1 → больше вес на Recall

β < 1 → больше вес на Precision

Ответ: β > 1

In [706]:
import pandas as pd
import numpy as np
from category_encoders import CountEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from sklearn.metrics import auc
from sklearn.model_selection import GridSearchCV

### 1.Download data from Don’tGetKicked competition.

In [707]:
data = pd.read_csv("training.csv")
data['PurchDate'] = pd.to_datetime(data['PurchDate'], errors='coerce')
data.sort_values('PurchDate', inplace=True)
data.reset_index(drop=True, inplace=True)

In [708]:
data.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,32389,0,2009-01-05,MANHEIM,2007,2,CHRYSLER,PACIFICA FWD 3.8L V6,Bas,4D SPORT,...,9906.0,11657.0,NaN,NaN,3453,80022,CO,6770.0,0,1389
1,32406,0,2009-01-05,MANHEIM,2005,4,FORD,FREESTAR FWD V6 3.9L,SES,4D PASSENGER 3.9L SES,...,5801.0,6949.0,NaN,NaN,22916,80022,CO,6160.0,0,941
2,32407,0,2009-01-05,MANHEIM,2004,5,DODGE,STRATUS 4C 2.4L I4 M,SE,4D SEDAN SE,...,4169.0,5114.0,NaN,NaN,3453,80022,CO,4250.0,0,1155
3,32408,0,2009-01-05,MANHEIM,2006,3,CHEVROLET,TRAILBLAZER EXT 4WD,LS,4D SUV 4.2L,...,10438.0,12158.0,NaN,NaN,22916,80022,CO,8180.0,0,1703
4,32409,0,2009-01-05,MANHEIM,2004,5,FORD,TAURUS 3.0L V6 EFI,SES,4D SEDAN SES DURATEC,...,4139.0,5351.0,NaN,NaN,22916,80022,CO,4900.0,0,825


### 2. Design the train/validation/test split.

In [709]:
num = (len(data)//3)
X_train = data.iloc[:num]
print(X_train.shape)
X_val   = data.iloc[num:2*num]
print(X_val.shape)
X_test  = data.iloc[2*num:]
print(X_test.shape)

(24327, 34)
(24327, 34)
(24329, 34)


In [710]:
y_train = X_train['IsBadBuy']
X_train = X_train.drop(columns=['IsBadBuy'])

y_val = X_val['IsBadBuy']
X_val = X_val.drop(columns=['IsBadBuy'])

y_test = X_test['IsBadBuy']
X_test = X_test.drop(columns=['IsBadBuy'])

In [711]:
print(f'train max - {X_train["PurchDate"].max()}', f'val min - {X_val["PurchDate"].min()}', sep="\n")
print(f'val max - {X_val["PurchDate"].max()}', f'test min - {X_test["PurchDate"].min()}', sep="\n")

train max - 2009-09-15 00:00:00
val min - 2009-09-15 00:00:00
val max - 2010-05-14 00:00:00
test min - 2010-05-14 00:00:00


train.PurchDate < valid.PurchDate < test.PurchDate

- сначала мы учимся на старых покупках

- потом проверяемся на более новых

- и в конце тестируемся на самых новых

In [712]:
X_train.head()

,RefId,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,32389,2009-01-05,MANHEIM,2007,2,CHRYSLER,PACIFICA FWD 3.8L V6,Bas,4D SPORT,BLUE,...,9906.0,11657.0,NaN,NaN,3453,80022,CO,6770.0,0,1389
1,32406,2009-01-05,MANHEIM,2005,4,FORD,FREESTAR FWD V6 3.9L,SES,4D PASSENGER 3.9L SES,SILVER,...,5801.0,6949.0,NaN,NaN,22916,80022,CO,6160.0,0,941
2,32407,2009-01-05,MANHEIM,2004,5,DODGE,STRATUS 4C 2.4L I4 M,SE,4D SEDAN SE,SILVER,...,4169.0,5114.0,NaN,NaN,3453,80022,CO,4250.0,0,1155
3,32408,2009-01-05,MANHEIM,2006,3,CHEVROLET,TRAILBLAZER EXT 4WD,LS,4D SUV 4.2L,WHITE,...,10438.0,12158.0,NaN,NaN,22916,80022,CO,8180.0,0,1703
4,32409,2009-01-05,MANHEIM,2004,5,FORD,TAURUS 3.0L V6 EFI,SES,4D SEDAN SES DURATEC,GOLD,...,4139.0,5351.0,NaN,NaN,22916,80022,CO,4900.0,0,825


### 3. Use LabelEncoder or OneHotEncoder from sklearn to preprocess categorical variables.

In [713]:
for column in data.columns:
    precent = ((data[column].isna().sum()) / (data[column].shape[0]))*100
    if precent:
        print(f'{column} - {precent}')

X_train = X_train.drop(columns=['PRIMEUNIT','AUCGUART'],axis=1)
X_val = X_val.drop(columns=['PRIMEUNIT','AUCGUART'],axis=1)
X_test = X_test.drop(columns=['PRIMEUNIT','AUCGUART'],axis=1)

Trim - 3.2336297493936947
SubModel - 0.010961456777605743
Color - 0.010961456777605743
Transmission - 0.01233163887480646
WheelTypeID - 4.342107066029075
WheelType - 4.348957976515079
Nationality - 0.00685091048600359
Size - 0.00685091048600359
TopThreeAmericanName - 0.00685091048600359
MMRAcquisitionAuctionAveragePrice - 0.02466327774961292
MMRAcquisitionAuctionCleanPrice - 0.02466327774961292
MMRAcquisitionRetailAveragePrice - 0.02466327774961292
MMRAcquisitonRetailCleanPrice - 0.02466327774961292
MMRCurrentAuctionAveragePrice - 0.4316073606182262
MMRCurrentAuctionCleanPrice - 0.4316073606182262
MMRCurrentRetailAveragePrice - 0.4316073606182262
MMRCurrentRetailCleanPrice - 0.4316073606182262
PRIMEUNIT - 95.31534740967075
AUCGUART - 95.31534740967075


Я проанализировала долю пропущенных значений по каждому признаку
Столбцы PRIMEUNIT и AUCGUART содержали более 95% пропусков,
поэтому я удалила их из train, validation и test, так как они не несут полезной информации для модели

In [714]:
def extract_date_features(df):
    df = df.copy()
    df['day'] = df['PurchDate'].dt.day
    df['month'] = df['PurchDate'].dt.month
    df = df.drop('PurchDate', axis=1)
    return df

X_train_object = extract_date_features(X_train)
X_val_object = extract_date_features(X_val)
X_test_object = extract_date_features(X_test)

In [715]:
object_columns = []
for column in X_train.columns:
    if ((X_train[column].dtypes))=='object':
        object_columns.append(column)
object_columns

['Auction',
 'Make',
 'Model',
 'Trim',
 'SubModel',
 'Color',
 'Transmission',
 'WheelType',
 'Nationality',
 'Size',
 'TopThreeAmericanName',
 'VNST']

In [716]:
encoder = CountEncoder(cols=object_columns,handle_missing='value')
encoder.fit(X_train_object)

,verbose,0
,cols,"['Auction', 'Make', ...]"
,drop_invariant,False
,return_df,True
,handle_unknown,'value'
,handle_missing,'value'
,min_group_size,None
,combine_min_nan_groups,True
,min_group_name,None
,normalize,False


In [717]:
X_train_enc = encoder.transform(X_train_object)
X_val_enc = encoder.transform(X_val_object)
X_test_enc = encoder.transform(X_test_object)

In [718]:
train_mean = X_train_enc.mean()
X_train_enc = X_train_enc.fillna(train_mean)
X_val_enc = X_val_enc.fillna(train_mean)
X_test_enc = X_test_enc.fillna(train_mean)

In [719]:
X_train_enc

,RefId,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost,day,month
0,32389,14099,2007,2,2774,99,4540,284,3432,23646,...,9906.0,11657.0,3453,80022,2016,6770.0,0,1389,5,1
1,32406,14099,2005,4,4264,309,265,46,5010,23646,...,5801.0,6949.0,22916,80022,2016,6160.0,0,941,5,1
2,32407,14099,2004,5,4784,250,3432,1427,5010,23646,...,4169.0,5114.0,3453,80022,2016,4250.0,0,1155,5,1
3,32408,14099,2006,3,5659,55,2988,177,4214,23646,...,10438.0,12158.0,22916,80022,2016,8180.0,0,1703,5,1
4,32409,14099,2004,5,4264,955,265,58,1890,23646,...,4139.0,5351.0,22916,80022,2016,4900.0,0,825,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24322,56894,14099,2005,4,5659,68,2988,4903,4214,23646,...,6341.0,7085.0,23359,92504,2554,4085.0,0,803,15,9
24323,56893,14099,2008,1,4784,276,3432,4903,3432,23646,...,11871.0,12554.0,23359,92504,2554,7555.0,0,1020,15,9
24324,56892,14099,2008,1,4784,276,3432,4903,2015,23646,...,9753.0,10571.0,23359,92504,2554,7555.0,0,1020,15,9
24325,15848,14099,2006,3,198,23,69,61,5010,23646,...,8697.0,10288.0,20207,77041,4562,7695.0,0,1272,15,9


In [720]:
X_train = X_train.drop(columns=['RefId', 'WheelTypeID'], errors='ignore')
X_val   = X_val.drop(columns=['RefId', 'WheelTypeID'], errors='ignore')
X_test  = X_test.drop(columns=['RefId', 'WheelTypeID'], errors='ignore')

Колонки RefId и WheelTypeID (идентификаторы) попадают в финальный датасет. RefId — просто порядковый номер строки, а WheelTypeID дублирует WheelType

In [721]:
X_train_enc.shape

(24327, 32)

### 4. Train LogisticRegression, GaussianNB, KNN

Для нормализации признаков использую StandardScaler, потому что признаки имеют разные масштабы,
а модели Logistic Regression и KNN чувствительны к масштабу.
MinMaxScaler менее устойчив к выбросам и хуже подходит при наличии one-hot признаков

In [722]:
X_train_enc.mean(axis=0), X_train_enc.std(axis=0)

(RefId                                37459.729519
 Auction                              10322.469067
 VehYear                               2004.823817
 VehicleAge                               4.176224
 Make                                  3505.839808
 Model                                  282.142023
 Trim                                  2019.137789
 SubModel                              1257.377441
 Color                                 3115.528713
 Transmission                         23002.684630
 WheelTypeID                              1.473977
 WheelType                            11144.603075
 VehOdo                               69842.535372
 Nationality                          18662.687097
 Size                                  4751.227977
 TopThreeAmericanName                  6915.567353
 MMRAcquisitionAuctionAveragePrice     5797.127611
 MMRAcquisitionAuctionCleanPrice       7028.352985
 MMRAcquisitionRetailAveragePrice      6753.751398
 MMRAcquisitonRetailCleanPrice 

In [723]:
scaler = StandardScaler()

scaler.fit(X_train_enc)

X_train = scaler.transform(X_train_enc)
X_val   = scaler.transform(X_val_enc)
X_test  = scaler.transform(X_test_enc)

In [724]:
X_train.mean(axis=0), X_train.std(axis=0)

(array([-1.49544901e-16, -1.86931126e-16, -6.66222534e-14, -9.34655631e-18,
         5.60793379e-17, -5.60793379e-17,  2.80396689e-17, -7.00991724e-17,
        -7.22897715e-17, -1.20336913e-16,  1.63564735e-16,  2.04455919e-16,
        -1.47208262e-16, -3.27129471e-17, -7.94457287e-17,  1.51881540e-16,
         3.45822584e-16, -3.73862253e-17, -1.16831954e-16, -1.49544901e-16,
         0.00000000e+00,  2.24317352e-16,  9.34655631e-17,  3.55169140e-16,
         4.67327816e-17, -9.69705218e-17, -1.72911292e-16, -1.35525067e-16,
         0.00000000e+00, -8.64556459e-17,  4.20595034e-17, -2.24317352e-16]),
 array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1.]))

In [725]:
X_train = pd.DataFrame(X_train,columns=list(X_train_enc.columns))
X_val = pd.DataFrame(X_val,columns=list(X_train_enc.columns))
X_test = pd.DataFrame(X_test,columns=list(X_train_enc.columns))

In [726]:
X_train.shape

(24327, 32)

Метрика GINI:

Gini=2⋅ROC-AUC−1

In [727]:
def gini_score(y_true, y_pred_proba):
    """
    y_true        — реальные метки (0/1)
    y_pred_proba  — предсказанные вероятности класса 1
    """
    return 2 * roc_auc_score(y_true, y_pred_proba) - 1

Для расчёта Gini используются вероятности класса, поэтому я применяю predict_proba, а не бинарные предсказания

In [728]:
logreg = LogisticRegression(solver='liblinear', n_jobs=1)
logreg.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [729]:
pred_proba_logreg = logreg.predict_proba(X_val)[:,1]
gini_score_log_sk = gini_score(y_val, pred_proba_logreg)

In [730]:
gaus = GaussianNB()
gaus.fit(X_train,y_train)

pred_proba_gaus = gaus.predict_proba(X_val)[:,1]
gini_score_naive_sk = gini_score(y_val, pred_proba_gaus)

In [731]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train,y_train)

pred_proba_knn = knn.predict_proba(X_val)[:,1]
gini_score_knn_sk = gini_score(y_val, pred_proba_knn)

In [732]:
print(f'{gini_score_log_sk} - logisticregr,\n{gini_score_naive_sk} - GaussianNB, \n{gini_score_knn_sk} - knn')

0.3794123011125552 - logisticregr,
0.4399117816062945 - GaussianNB, 
0.31862963549495693 - knn


Все модели превышают требуемый порог 0.15.

### 5. Implement Gini score calculation

ROC AUC — это метрика бинарной классификации, которая показывает, насколько хорошо модель различает два класса.
Её можно интерпретировать как вероятность того, что случайный объект класса 1 получит более высокую предсказанную вероятность, чем случайный объект класса 0.
Базовое значение ROC AUC — 0.5 (случайная модель), идеальное — 1.

Gini — это линейное преобразование ROC AUC:

Gini=2⋅ROC AUC−1

Если модель случайная, Gini равен 0, если идеальная — 1.
Gini часто используется в задачах риск-моделирования и скоринга.

In [733]:
def gini_score(y_true, y_pred):
    """
    Вычисляет Gini коэффициент.

    y_true : массив 0/1 (правильные метки)
    y_pred : массив вероятностей предсказания класса 1

    Возвращает:
    gini : float
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    order = np.argsort(-y_pred)
    y_true_sorted = y_true[order]

    n_pos = np.sum(y_true_sorted)
    n_neg = len(y_true_sorted) - n_pos

    if n_pos == 0 or n_neg == 0:
        return 0 

    cum_pos = np.cumsum(y_true_sorted)
    total_pos_rank = np.sum(cum_pos[y_true_sorted==0]) 

    # ROC AUC
    auc = total_pos_rank / (n_pos * n_neg)

    # Gini
    gini = 2 * auc - 1
    return gini

In [734]:
models = {
    "LogReg": logreg,
    "GaussianNB": gaus,
    "KNN": knn
}

for name, model in models.items():
    y_val_proba = model.predict_proba(X_val)[:, 1]

    gini_my = gini_score(y_val, y_val_proba)
    gini_sk = 2 * roc_auc_score(y_val, y_val_proba) - 1

    print(f"{name}: my_gini={gini_my:.4f}, sklearn_gini={gini_sk:.4f}")


LogReg: my_gini=0.3794, sklearn_gini=0.3794
GaussianNB: my_gini=0.4399, sklearn_gini=0.4399
KNN: my_gini=0.3214, sklearn_gini=0.3186


### 6. Implement your own versions of LogisticRegression, KNN and NaiveBayes classifiers

Я реализовала собственные версии Logistic Regression, KNN и Naive Bayes.
Для Logistic Regression использовался SGD с вычислением градиентов функции потерь.
Качество моделей сопоставимо с результатами sklearn, что подтверждает корректность реализации

Logistic Regression

In [735]:
class MyLogisticRegression:
    def __init__(self,learn_rate = 0.01,n_epochs=5000,random_state=None):
        self.learn_rate = learn_rate
        self.n_epochs = n_epochs
        self.random_state = random_state

    def sigmoida(self,x):
        return 1 / (1+np.exp(-x))

    def fit(self,X,y):
        X = np.array(X)
        y = np.array(y).flatten()

        X = np.c_[np.ones(X.shape[0]),X]
        n_samples = X.shape[0]
        n_features = X.shape[1]

        if self.random_state is not None:
            np.random.seed(self.random_state)
        self.w = np.zeros(n_features)

        for _ in range(self.n_epochs):
            preds = self.sigmoida(X @ self.w)
            grad = (1/n_samples) * (X.T @ (preds - y))
            self.w -= self.learn_rate * grad

    def predict_proba(self, X_test):
        X = np.array(X_test)
        X = np.c_[np.ones(X.shape[0]), X]
        return self.sigmoida(X @ self.w)

    def predict(self, X_test):
        X = np.array(X_test)
        X = np.c_[np.ones(X.shape[0]), X]
        y_proba = self.sigmoida(X @ self.w)
        return np.where(y_proba >= 0.5, 1, 0)

In [736]:
my_lr = MyLogisticRegression(learn_rate = 0.01,n_epochs=5000,random_state=21)
my_lr.fit(X_train, y_train)

KNN —
для каждой точки считаем расстояния до train

берём k ближайших

считаем среднее значение y

In [737]:
class MyKNN:
    def __init__(self,k_neighbours):
        self.k_neighbours = k_neighbours

    def fit(self,X,y):
        self.X = np.asarray(X, dtype=np.float64)
        self.y = np.asarray(y)


    # Векторизованный predict_proba
    def predict_proba(self, X_test):
        X = np.asarray(X_test, dtype=np.float64)
        dists = np.linalg.norm(X[:, None] - self.X[None, :], axis=2)
        k_idx = np.argsort(dists, axis=1)[:, :self.k_neighbours]
        return self.y[k_idx].mean(axis=1)

    def predict(self, X_test):
        proba = self.predict_proba(X_test)
        return (proba >= 0.5).astype(int)

Naive Bayes —

для каждого класса считаем:

mean

variance

считаем likelihood по формуле Гаусса

In [738]:
class MyGaussianNB:
    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)

        self.classes_, counts = np.unique(y, return_counts=True)
        self.priors = counts / len(y)
        self.n_classes = len(self.classes_)

        self.means_ = np.array([np.mean(X[y == c], axis=0) for c in self.classes_])
        self.stds_ = np.array([np.std(X[y == c], axis=0) for c in self.classes_])
        self.stds_ = np.clip(self.stds_, 1e-9, None)

    def formula(self, x, mean, std):
        return -0.5 * (np.log(2 * np.pi) + 2 * np.log(std) + ((x - mean) / std) ** 2)

    def predict_proba(self, X_test):
        X = np.asarray(X_test, dtype=np.float64)
        n_samples = X.shape[0]
        log_posteriors = np.zeros((n_samples, self.n_classes))

        for i in range(n_samples):
            x = X[i]
            for k in range(self.n_classes):
                log_likelihood = np.sum(self.formula(x, self.means_[k], self.stds_[k]))
                log_posteriors[i, k] = np.log(self.priors[k]) + log_likelihood

        log_posteriors -= np.max(log_posteriors, axis=1, keepdims=True)
        prob = np.exp(log_posteriors)
        prob /= np.sum(prob, axis=1, keepdims=True)
        return prob

    def predict(self, X_test):
        return self.classes_[np.argmax(self.predict_proba(X_test), axis=1)]

In [739]:
my_nb = MyGaussianNB()
my_nb.fit(X_train, y_train)

In [740]:
lr_proba = my_lr.predict_proba(X_val)

In [741]:
auc_lr = roc_auc_score(y_val, lr_proba)
print("AUC LR:", auc_lr)

AUC LR: 0.7220389561893708


In [742]:
gini_score_log_my = 2 * auc_lr - 1
print("Gini LR:", gini_score_log_my)

Gini LR: 0.4440779123787415


In [743]:
nb_proba = my_nb.predict_proba(X_val)[:, 1] 
gini_score(y_val, nb_proba)                   

0.4256939976999907

In [744]:
my_knn = MyKNN(k_neighbours=5)
my_knn.fit(X_train[:3000], y_train.values[:3000])
knn_proba = my_knn.predict_proba(X_val[:1000])
gini_knn_my = gini_score(y_val.values[:1000], knn_proba)

In [745]:
gini_score_log_my = gini_score(y_val, lr_proba)
gini_score_naive_my = gini_score(y_val, nb_proba)
gini_score_knn_my = gini_score(y_val[:5000], knn_proba)

In [746]:
print("Logistic Regression")
print("sklearn:", gini_score_log_sk)
print("mine   :", gini_score_log_my)

print("\nGaussianNB")
print("sklearn:", gini_score_naive_sk)
print("mine   :", gini_score_naive_my)

print("\nKNN")
print("sklearn:", gini_score_knn_sk)
print("mine   :", gini_score_knn_my)

Logistic Regression
sklearn: 0.3794123011125552
mine   : 0.4440779123787413

GaussianNB
sklearn: 0.4399117816062945
mine   : 0.4256939976999907

KNN
sklearn: 0.31862963549495693
mine   : 0.29665667891967584


Собственные реализации моделей показывают сопоставимые результаты с реализацией sklearn. Небольшие различия объясняются различиями в оптимизации и реализации алгоритмов

### 7. Try to create non-linear features

fraction-признаки

In [747]:
print(type(X_train))  
print(X_train.columns)

<class 'pandas.core.frame.DataFrame'>
Index(['RefId', 'Auction', 'VehYear', 'VehicleAge', 'Make', 'Model', 'Trim',
       'SubModel', 'Color', 'Transmission', 'WheelTypeID', 'WheelType',
       'VehOdo', 'Nationality', 'Size', 'TopThreeAmericanName',
       'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice',
       'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice',
       'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice',
       'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'BYRNO',
       'VNZIP1', 'VNST', 'VehBCost', 'IsOnlineSale', 'WarrantyCost', 'day',
       'month'],
      dtype='object')


In [748]:
def add_extra_features_grouped(X_train, X_val, X_test):
    group_cols = ['Auction', 'Make', 'Nationality', 'Size', 'VNST']
    num_cols = ['VehBCost', 'WarrantyCost', 'VehicleAge']

    X_train = X_train.copy()
    X_val   = X_val.copy()
    X_test  = X_test.copy()

    for gc in group_cols:
        for nc in num_cols:
            col_name = f'{gc}Mean{nc}'
            train_means = X_train.groupby(gc)[nc].mean()

            X_train[col_name] = X_train[gc].map(train_means)
            X_val[col_name]   = X_val[gc].map(train_means)
            X_test[col_name]  = X_test[gc].map(train_means)

    return X_train, X_val, X_test

X_train, X_val, X_test = add_extra_features_grouped(X_train, X_val, X_test)

In [749]:
def add_extra_features(X):
    X = X.copy()
    X['WarrantyToCost'] = X['WarrantyCost'] / (X['VehBCost'] + 1e-6)
    X['cost_to_retail_ratio'] = X['VehBCost'] / (X['MMRCurrentRetailAveragePrice'] + 1e-6)
    return X

In [750]:
print('Auction' in X_train.columns)
print(X_train.columns)

True
Index(['RefId', 'Auction', 'VehYear', 'VehicleAge', 'Make', 'Model', 'Trim',
       'SubModel', 'Color', 'Transmission', 'WheelTypeID', 'WheelType',
       'VehOdo', 'Nationality', 'Size', 'TopThreeAmericanName',
       'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice',
       'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice',
       'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice',
       'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'BYRNO',
       'VNZIP1', 'VNST', 'VehBCost', 'IsOnlineSale', 'WarrantyCost', 'day',
       'month', 'AuctionMeanVehBCost', 'AuctionMeanWarrantyCost',
       'AuctionMeanVehicleAge', 'MakeMeanVehBCost', 'MakeMeanWarrantyCost',
       'MakeMeanVehicleAge', 'NationalityMeanVehBCost',
       'NationalityMeanWarrantyCost', 'NationalityMeanVehicleAge',
       'SizeMeanVehBCost', 'SizeMeanWarrantyCost', 'SizeMeanVehicleAge',
       'VNSTMeanVehBCost', 'VNSTMeanWarrantyCost', 'VNSTMeanVehi

In [751]:
X_train = add_extra_features(X_train)
X_val   = add_extra_features(X_val)
X_test  = add_extra_features(X_test)

In [752]:
from category_encoders import CountEncoder

object_columns = X_train.select_dtypes(include=['object']).columns.tolist()

encoder = CountEncoder(cols=object_columns, handle_missing='value')
encoder.fit(X_train)
X_train = encoder.transform(X_train)
X_val   = encoder.transform(X_val)
X_test  = encoder.transform(X_test)

In [753]:
train_mean = X_train.mean()
X_train_new = X_train.fillna(train_mean)
X_val_new = X_val.fillna(train_mean)
X_test_new = X_test.fillna(train_mean)

In [754]:
scaler = MinMaxScaler()
scaler.fit(X_train_new)

,feature_range,"(0, ...)"
,copy,True
,clip,False


In [755]:
X_train_scaled = scaler.transform(X_train_new)
X_val_scaled = scaler.transform(X_val_new)
X_test_scaled = scaler.transform(X_test_new)

X_train = X_train_scaled
X_val = X_val_scaled
X_test = X_test_scaled

In [756]:
print(data.columns)


Index(['RefId', 'IsBadBuy', 'PurchDate', 'Auction', 'VehYear', 'VehicleAge',
       'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission',
       'WheelTypeID', 'WheelType', 'VehOdo', 'Nationality', 'Size',
       'TopThreeAmericanName', 'MMRAcquisitionAuctionAveragePrice',
       'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice',
       'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice',
       'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice',
       'MMRCurrentRetailCleanPrice', 'PRIMEUNIT', 'AUCGUART', 'BYRNO',
       'VNZIP1', 'VNST', 'VehBCost', 'IsOnlineSale', 'WarrantyCost'],
      dtype='object')


In [757]:
Logreg = LogisticRegression(max_iter=5000,random_state=21)
Logreg.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,21
,solver,'lbfgs'
,max_iter,5000
,multi_class,'deprecated'


In [758]:
pred_proba_logreg_new = Logreg.predict_proba(X_val)[:,1]
gini_score_log_sk_new  = gini_score(y_val, pred_proba_logreg_new)

In [759]:
Naive = GaussianNB()
Naive.fit(X_train,y_train)

,priors,None
,var_smoothing,1e-09


In [760]:
pred_proba_gaus_new = Naive.predict_proba(X_val)[:,1]
gini_score_naive_sk_new = gini_score(y_val, pred_proba_gaus_new)

In [761]:
Knn_new = KNeighborsClassifier()
Knn_new.fit(X_train,y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [762]:
pred_proba_knn_new = Knn_new.predict_proba(X_val)[:,1]
gini_score_knn_sk_new = gini_score(pred_proba_knn_new,y_val)

In [763]:
logreg = LogisticRegression(max_iter=5000, random_state=21)
logreg.fit(X_train_scaled, y_train)
pred_proba_logreg = logreg.predict_proba(X_val_scaled)[:,1]

In [764]:
gaus = GaussianNB()
gaus.fit(X_train_scaled, y_train)
pred_proba_gaus = gaus.predict_proba(X_val_scaled)[:,1]

In [765]:
knn = KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)
pred_proba_knn = knn.predict_proba(X_val_scaled)[:,1]

In [766]:
gini_logreg_sk = gini_score(y_val, pred_proba_logreg)
gini_gaus_sk   = gini_score(y_val, pred_proba_gaus)
gini_knn_sk    = gini_score(y_val, pred_proba_knn)

In [767]:
print("Sklearn models Gini:")
print(f"Logistic Regression: {gini_logreg_sk:.4f}")
print(f"GaussianNB: {gini_gaus_sk:.4f}")
print(f"KNN: {gini_knn_sk:.4f}")

Sklearn models Gini:
Logistic Regression: 0.4378
GaussianNB: 0.3850
KNN: 0.2960


In [768]:
my_lr = MyLogisticRegression(learn_rate=0.01, n_epochs=5000, random_state=21)
my_lr.fit(X_train_scaled, y_train)
pred_proba_lr_my = my_lr.predict_proba(X_val_scaled)

In [769]:
my_nb = MyGaussianNB()
my_nb.fit(X_train_scaled, y_train)
pred_proba_nb_my = my_nb.predict_proba(X_val_scaled)[:, 1]

In [770]:
my_knn = MyKNN(k_neighbours=5)
my_knn.fit(X_train[:3000], y_train.values[:3000])
pred_proba_knn_my = my_knn.predict_proba(X_val_scaled[:1000])
gini_knn_my = gini_score(y_val.values[:1000], pred_proba_knn_my)

In [771]:
gini_lr_my   = gini_score(y_val, pred_proba_lr_my)
gini_nb_my   = gini_score(y_val, pred_proba_nb_my)
gini_knn_my  = gini_score(y_val, pred_proba_knn_my)

print("My models Gini:")
print(f"Logistic Regression: {gini_lr_my:.4f}")
print(f"GaussianNB: {gini_nb_my:.4f}")
print(f"KNN: {gini_knn_my:.4f}")

My models Gini:
Logistic Regression: 0.4088
GaussianNB: 0.3725
KNN: 0.2347


In [772]:
models = {
    "LogReg": (pred_proba_logreg, pred_proba_lr_my),
    "GaussianNB": (pred_proba_gaus, pred_proba_nb_my),
    "KNN": (pred_proba_knn, pred_proba_knn_my)
}

for name, (sk_pred, my_pred) in models.items():
    gini_sk = gini_score(y_val, sk_pred)
    gini_my = gini_score(y_val, my_pred)
    print(f"{name}: Sklearn Gini = {gini_sk:.4f}, My Gini = {gini_my:.4f}")

LogReg: Sklearn Gini = 0.4378, My Gini = 0.4088
GaussianNB: Sklearn Gini = 0.3850, My Gini = 0.3725
KNN: Sklearn Gini = 0.2960, My Gini = 0.2347


### 8. Determine the best features for the problem using the coefficients of the logistic model. Try to eliminate useless features by hand and by L1 regularization

In [773]:
features_names = X_train_new.columns.tolist()

In [774]:
coefs_np = np.array(Logreg.coef_)[0]

coefs = pd.DataFrame({
    'ferature' : features_names,
    'coefs' : abs(coefs_np)
}).sort_values(by='coefs',ascending=False)

In [775]:
coefs

,ferature,coefs
11,WheelType,3.518591
27,VehBCost,2.966922
10,WheelTypeID,1.771221
12,VehOdo,1.048464
46,VNSTMeanVehicleAge,1.044745
39,NationalityMeanWarrantyCost,1.027702
3,VehicleAge,0.970083
24,BYRNO,0.893246
29,WarrantyCost,0.839543
23,MMRCurrentRetailCleanPrice,0.818183


In [776]:
top_features = coefs[:25]['ferature'].tolist()
top_features

['WheelType',
 'VehBCost',
 'WheelTypeID',
 'VehOdo',
 'VNSTMeanVehicleAge',
 'NationalityMeanWarrantyCost',
 'VehicleAge',
 'BYRNO',
 'WarrantyCost',
 'MMRCurrentRetailCleanPrice',
 'MMRCurrentRetailAveragePrice',
 'NationalityMeanVehBCost',
 'MakeMeanVehBCost',
 'VehYear',
 'Model',
 'MakeMeanWarrantyCost',
 'TopThreeAmericanName',
 'VNSTMeanWarrantyCost',
 'Transmission',
 'month',
 'AuctionMeanVehicleAge',
 'Nationality',
 'SizeMeanVehBCost',
 'MMRCurrentAuctionAveragePrice',
 'SubModel']

In [777]:
X_train = pd.DataFrame(X_train,columns=list(X_train_new.columns)) 
X_val = pd.DataFrame(X_val,columns=list(X_train_new.columns)) 
X_test = pd.DataFrame(X_test,columns=list(X_train_new.columns))

In [778]:
X_train_top = X_train[top_features] 
X_val_top = X_val[top_features] 
X_test_top = X_test[top_features]

In [779]:
type(X_train)

pandas.core.frame.DataFrame

In [780]:
logreg_top = LogisticRegression(max_iter=1000)
logreg_top.fit(X_train_top, y_train)

proba_top = logreg_top.predict_proba(X_val_top)[:,1]
gini_manual = 2 * roc_auc_score(y_val, proba_top) - 1

print("Gini manual selection:", gini_manual)

Gini manual selection: 0.4422558919441313


L1 regularization

In [781]:
logreg_l1 = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=1.0,
    max_iter=1000
)

logreg_l1.fit(X_train_new, y_train)

proba_l1 = logreg_l1.predict_proba(X_val_new)[:,1]
gini_l1 = 2 * roc_auc_score(y_val, proba_l1) - 1

print("Gini L1:", gini_l1)

Gini L1: 0.43549242401388333


Ручной отбор признаков и L1-регуляризация позволили сократить число признаков при минимальной потере качества (по метрике Gini). Базовая модель показала немного лучший результат, однако L1-регуляризация является более удобным и устойчивым способом отбора признаков

### 9. Select your best model (algorithm + feature set) and tweak its hyperparameters to increase the Gini score on the validation dataset

Нстройка гиперпараметров

C = 1/λ

где λ — сила регуляризации (штрафует модель за большие коэффициенты)

маленький C → сильная регуляризация

большой C → слабая

In [782]:
param_grid = {'C': np.logspace(-4, 2, 50)}

model = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    random_state=21
)

grid = GridSearchCV(
    model,
    param_grid,
    scoring='roc_auc',
    cv=5
)

grid.fit(X_train_top, y_train)

print("Best C:", grid.best_params_['C'])
print("Best Gini:", 2 * grid.best_score_ - 1)

Best C: 0.8286427728546842
Best Gini: 0.4937007030978955


В результате подбора гиперпараметров с использованием GridSearchCV было найдено оптимальное значение параметра C ≈ 0.83.
Наибольшее влияние на качество модели оказывает именно параметр C, так как он регулирует степень регуляризации.
Подобранное значение позволило увеличить Gini до 0.49, что выше предыдущих результатов

In [783]:
best_model = LogisticRegression(
    C=0.8286427728546842,
    penalty='l1',
    solver='liblinear',
    random_state=21
)

best_model.fit(X_train_top, y_train)

val_proba = best_model.predict_proba(X_val_top)[:,1]
val_gini = 2 * roc_auc_score(y_val, val_proba) - 1

print("Validation Gini:", val_gini)

Validation Gini: 0.4436141709766699


В собственной реализации логистической регрессии регуляризация не использовалась. В модели sklearn применяется регуляризация по умолчанию, управляемая параметром C=1.0 (и penalty='l2'). Подбор данного параметра позволил улучшить обобщающую способность модели и повысить значение Gini на валидационной выборке.

Параметр C имеет наибольшее влияние, так как он контролирует степень регуляризации и баланс между переобучением и недообучением модели.

После настройки гиперпараметров значение Gini на валидационной выборке увеличилось до 0.444, что свидетельствует об улучшении обобщающей способности модели. Модель немного подстроилась под train, но на новых данных работает слабее

### 10. Check the Gini scores on all three datasets for your best model: training Gini, valid Gini, test Gini

In [784]:
train_proba = best_model.predict_proba(X_train_top)[:,1]
val_proba   = best_model.predict_proba(X_val_top)[:,1]
test_proba  = best_model.predict_proba(X_test_top)[:,1]

In [785]:
gini_score_train = gini_score(y_train, train_proba)
gini_score_val   = gini_score(y_val, val_proba)
gini_score_test  = gini_score(y_test, test_proba)

In [786]:
print(f'All metrics on 3 datasets:\n'
      f'{gini_score_train} - train\n'
      f'{gini_score_val} - validation\n'
      f'{gini_score_test} - test')

All metrics on 3 datasets:
0.5009210114847278 - train
0.4436141709766699 - validation
0.4811117963670901 - test


Gini на обучающей выборке (0.501) выше, чем на валидационной (0.444), что говорит о небольшом переобучении модели. Однако на тестовой выборке Gini (0.481) остаётся близким к значению на обучении и не падает по сравнению с валидацией. Это свидетельствует о том, что модель хорошо обобщает данные и не является сильно переобученной

### 11. Implement calculation of Recall, Precision, F1 score and AUC PR metrics.
Compare your algorithms on the test dataset using AUC PR metric.

Precision (точность)

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

TP = True Positives (правильно предсказанные 1)

FP = False Positives (неверно предсказанные 1)

Доля правильно предсказанных положительных объектов среди всех объектов, предсказанных как положительные.

Recall (полнота)

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

FN = False Negatives (неверно пропущенные 1)

Доля правильно предсказанных положительных объектов среди всех реальных положительных объектов.

F1 Score

$$
F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

Гармоническое среднее Precision и Recall, баланс между ними.

Если Precision высокое, но Recall низкий → F1 будет средним

Если обе метрики высокие → F1 высокое

Average Precision (AUC PR)

$$
\text{AUC PR} = \int_0^1 P(R) \, dR
$$

Где 
P(R)
P(R) — Precision при значении Recall = 
R
R.

Площадь под кривой Precision-Recall при всех возможных порогах.
Используется для оценки качества моделей на несбалансированных данных, когда положительный класс редкий.

| Метрика | Что учитывает       | Когда полезно                                |
| ------- | ------------------- | -------------------------------------------- |
| ROC AUC | TP rate vs FP rate  | Общая способность отделять классы            |
| PR AUC  | Precision vs Recall | Редкие положительные примеры, важна точность |


In [787]:
def calculate_metrics(y_true, y_pred_proba):
    y_true = np.asarray(y_true)
    y_pred_proba = np.asarray(y_pred_proba)
    
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    tp = np.sum((y_true == 1)&(y_pred == 1))
    fp = np.sum((y_true == 0)&(y_pred == 1))
    fn = np.sum((y_true == 1)&(y_pred == 0))
    
    recall = tp/(tp + fn)
    precision = tp/(tp + fp)
    f1 = 2 * (precision * recall)/(precision + recall)

    n_pos = np.sum(y_true)
    desc_idx = np.argsort(-y_pred_proba, kind="mergesort")
    y_true_sorted = y_true[desc_idx]
    
    # Накопленные tp и fp
    tp_cum = np.cumsum(y_true_sorted)
    fp_cum = np.cumsum(1 - y_true_sorted)
    precision_curve = tp_cum / (tp_cum + fp_cum)
    recall_curve = tp_cum / n_pos
    
    #начальная точка
    precision_curve = np.concatenate([[1.0], precision_curve])
    recall_curve = np.concatenate([[0.0], recall_curve])

    auc_pr = np.sum(precision_curve[1:] * (recall_curve[1:] - recall_curve[:-1]))

    return {
        'recall': float(recall),
        'precision': float(precision),
        'f1': float(f1),
        'auc_pr': float(auc_pr)
    }

In [788]:
def check_metrics(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:,1]
    return calculate_metrics(y_test, y_proba)

In [789]:
best_lr = LogisticRegression(
    solver='liblinear', penalty='l1', C=grid.best_params_['C'], random_state=21
)
my_nb = GaussianNB()
my_knn = KNeighborsClassifier()

In [790]:
print("Logistic Regression (топ-25 фич):")
lr_metrics = check_metrics(best_lr, X_train_top, y_train, X_test_top, y_test)
print(lr_metrics)

print("\nGaussianNB (топ-25 фич):")
nb_metrics = check_metrics(my_nb, X_train_top, y_train, X_test_top, y_test)
print(nb_metrics)

print("\nKNN (топ-25 фич):")
knn_metrics = check_metrics(my_knn, X_train_top, y_train, X_test_top, y_test)
print(knn_metrics)

Logistic Regression (топ-25 фич):
{'recall': 0.243870112657389, 'precision': 0.7763713080168776, 'f1': 0.3711548159354514, 'auc_pr': 0.43133845036417495}

GaussianNB (топ-25 фич):
{'recall': 0.4042412193505633, 'precision': 0.3634197199880846, 'f1': 0.3827450980392157, 'auc_pr': 0.37981267676626307}

KNN (топ-25 фич):
{'recall': 0.2624254473161034, 'precision': 0.5819250551065394, 'f1': 0.36172642155743323, 'auc_pr': 0.37748306092043443}


### 12. Which hard label metric do you prefer for the task of detecting "lemon" cars?

Для задачи обнаружения «lemon» автомобилей (то есть класса IsBadBuy = 1), если мы говорим про hard label metric (то есть метрику, которая оценивает предсказания как 0 или 1, а не вероятности), важно учитывать, что:

Класс плохих машин обычно редкий (имбаланс классов).

Ошибка пропуска плохого автомобиля (FN) может стоить дорого, а ошибка на нормальной машине (FP) — менее критична.

Лучше всего использовать F1-score или Recall, если задача – минимизировать риск покупки плохой машины.

Recall особенно важен, потому что пропущенный «лимон» (FN) может стоить дорого.

Если хочется компромисс между ложными срабатываниями и пропущенными лимонами — F1-score.

Если цель — не пропустить лимонные машины, лучше выбирать GaussianNB, потому что у него самый высокий Recall, несмотря на низкий Precision.